# Weather Trend Forecasting

This project analyzes the Global Weather Repository dataset to explore weather patterns across locations and build a model for next-day temperature forecasting.

The analysis includes data cleaning, exploratory analysis, feature engineering, forecasting models, and evaluation.

## 1. Data Loading and Initial Inspection

In [ ]:
!pip install -q kaggle
!pip install catboost

In [ ]:
import os
from getpass import getpass

In [ ]:
os.environ["Kaggle_API_TOKEN"] = getpass("Api token : ")

In [ ]:
!kaggle datasets download -d nelgiriyewithana/global-weather-repository
!unzip -o global-weather-repository.zip

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
import shap
import joblib

In [ ]:
df = pd.read_csv("GlobalWeatherRepository.csv")
print(df.head())
print("Shape: ", df.shape)

In [ ]:
df.info()

### Feature Notes

A few variables need some context before analysis:

- `pressure_mb`: atmospheric pressure in millibars.
- `precip_mm`: precipitation recorded for the observation period, not a daily rainfall total.
- `visibility_km`: reported visibility distance.
- `uv_index`: UV intensity.
- Air-quality columns contain pollutant measurements and EPA/DEFRA air-quality indices.



---



---



In [ ]:
num = df.select_dtypes(exclude='O')
print(num.columns)
ob = df.select_dtypes(include='O')
print(ob.columns)

In [ ]:
num.iloc[:,0:15].describe()

In [ ]:
num.iloc[:,15:].describe()

### Initial Data Quality Checks

The summary statistics suggest two main issues to investigate:

- possible invalid or extreme values in several numeric features
- duplicate measurements stored in different units.

These are checked before the main exploratory analysis.



---



---



## 2. Data Cleaning

In [ ]:
# -9999 flag
(df == -9999).sum()[lambda x: x > 0]

In [ ]:
cols = [
    "air_quality_Carbon_Monoxide",
    "air_quality_Sulphur_dioxide"
]

df.loc[
    df[cols].eq(-9999).any(axis=1),
    [
        "country",
        "location_name",
        "last_updated",
        "air_quality_Carbon_Monoxide",
        "air_quality_Sulphur_dioxide",
        "air_quality_PM2.5",
        "air_quality_PM10"
    ]
]

In [ ]:
df[cols] = df[cols].replace(-9999, np.nan)

- Two air-quality values used `-9999` as a missing-value marker. They were replaced with `NaN` so they would not be treated as real measurements.



---



---



In [ ]:
# Redundant columns
pairs = [
    ("temperature_celsius", "temperature_fahrenheit"),
    ("feels_like_celsius", "feels_like_fahrenheit"),
    ("wind_kph", "wind_mph"),
    ("gust_kph", "gust_mph"),
    ("pressure_mb", "pressure_in"),
    ("precip_mm", "precip_in"),
    ("visibility_km", "visibility_miles")
]

for col1, col2 in pairs:
    print(
        col1, "XX", col2, "->>",
        round(df[[col1, col2]].corr().iloc[0, 1], 6)
    )

- The metric and original versions contain the same information apart from rounding, so the original duplicates are removed.

In [ ]:
drop_cols = [
    "temperature_fahrenheit",
    "feels_like_fahrenheit",
    "wind_mph",
    "gust_mph",
    "pressure_in",
    "precip_in",
    "visibility_miles"
]

df.drop(columns=drop_cols, inplace=True)

In [ ]:
df.columns.to_list()



---



---



In [ ]:
print(df[["last_updated", "last_updated_epoch"]].head())
print("\nUTC Last Updated Epoch\n",
    pd.to_datetime(
        df["last_updated_epoch"].head(),
        unit="s",
        utc=True
    )
)

dates = pd.to_datetime(df["last_updated"])
print("\nDate range:", dates.min(), "to", dates.max())

`last_updated` records the local observation time, while `last_updated_epoch` can be converted to a consistent UTC timestamp.

Both are useful: local time for seasonal/calendar features and UTC time for measuring elapsed time between observations.

In [ ]:
df["last_updated"] = pd.to_datetime(df["last_updated"])
df["last_updated_utc"] = pd.to_datetime(
    df["last_updated_epoch"],
    unit="s",
    utc=True
)

df.drop(columns="last_updated_epoch", inplace=True)

In [ ]:
# sorted(df["country"].unique())

In [ ]:
country_replacements = {
    "Bélgica": "Belgium",
    "Estonie": "Estonia",
    "Inde": "India",
    "Jemen": "Yemen",
    "Komoren": "Comoros",
    "Letonia": "Latvia",
    "Malásia": "Malaysia",
    "Marrocos": "Morocco",
    "Mexique": "Mexico",
    "Polônia": "Poland",
    "Saint-Vincent-et-les-Grenadines": "Saint Vincent and the Grenadines",
    "Saudi Arabien": "Saudi Arabia",
    "Südkorea": "South Korea",
    "Turkménistan": "Turkmenistan",
    "USA United States of America": "United States of America",
    "Гватемала": "Guatemala",
    "Польша": "Poland",
    "Турция": "Turkey",
    "كولومبيا": "Colombia",
    "火鸡": "Turkey"
}
print("Countries before:", df["country"].nunique())
df["country"] = df["country"].replace(country_replacements)
print("Countries after:", df["country"].nunique())

### Country Names

Several countries appeared under different languages or formats. These labels were standardized, reducing the number of unique country labels from **211 to 191**.



---



---



### Anomaly Screening

The IQR method was used as an initial screening tool to identify features worth inspecting. Values were not removed automatically because extreme weather observations can still be valid.

In [ ]:
num = df.select_dtypes(exclude='O')
outlier_columns = []
for col in num.columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    outlier_columns.append({
        "column": col,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": len(outliers),
        "min": df[col].min(),
        "max": df[col].max()
    })

outlier_columns = pd.DataFrame(outlier_columns)

outlier_columns.sort_values(
    "outlier_count",
    ascending=False
)

The main features requiring further inspection were `temperature`, `wind` and `gust speed`, `pressure`, `precipitation`, and `air-quality measurements`. `Coordinates`, `timestamps`, and discrete air-quality indices were not treated as continuous outliers.

### Weather Variables

In [ ]:
df["temperature_celsius"].describe()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x="temperature_celsius", bins=60, ax=ax[0])
ax[0].set_title("Temperature Distribution")
ax[0].set_xlabel("Temperature (°C)")
ax[0].set_ylabel("Frequency")

sns.boxplot(data=df, x="temperature_celsius", ax=ax[1])
ax[1].set_title("Temperature Boxplot")
ax[1].set_xlabel("Temperature (°C)")

plt.tight_layout()
plt.show()

In [ ]:
df.nlargest(10, "temperature_celsius")[
    [
        "country",
        "location_name",
        "last_updated",
        "temperature_celsius",
        "feels_like_celsius",
        "condition_text",
        "humidity",
        "pressure_mb"
    ]
]

In [ ]:
df[df["location_name"] == "Suva"].nlargest(10, "temperature_celsius")[
    [
        "last_updated",
        "temperature_celsius",
        "feels_like_celsius",
        "humidity",
        "pressure_mb",
        "condition_text"
    ]
]

The 79.3°C observation in Suva was inconsistent with the surrounding temperature and weather conditions and was treated as a data-entry error.

In [ ]:
df = df[
    ~(
        (df["country"] == "Fiji Islands")
        & (df["location_name"] == "Suva")
        & (df["last_updated"] == "2026-04-24 18:30:00")
        & (df["temperature_celsius"] == 79.3)
    )
].copy()



---



In [ ]:
df["feels_like_celsius"].describe()

`feels_like_celsius` contained extreme values but no observations with enough evidence to classify them as data errors, so they were retained.



---



In [ ]:
df[["wind_kph", "gust_kph"]].describe()

In [ ]:
df.nlargest(10, "wind_kph")[
    [
        "country",
        "location_name",
        "last_updated",
        "wind_kph",
        "gust_kph",
        "condition_text"
    ]
]

In [ ]:
df[
    df["location_name"].isin(
        ["Bujumbura", "Addis Ababa", "Basseterre", "Suva"]
    )].groupby("location_name")[["wind_kph", "gust_kph"]].agg(["min", "median", "max"])

In [ ]:
df.loc[
    [7601, 7248, 1193, 8675, 834],
    [
        "country",
        "location_name",
        "last_updated",
        "wind_kph",
        "gust_kph",
        "wind_degree",
        "pressure_mb",
        "precip_mm",
        "humidity",
        "condition_text"
    ]
]

In [ ]:
bad_wind = (
    ((df["location_name"] == "Bujumbura") & (df["wind_kph"].isin([2963.2, 258.8])))
    | ((df["location_name"] == "Addis Ababa") & (df["wind_kph"] == 272.2))
    | ((df["location_name"] == "Basseterre") & (df["wind_kph"] == 205.9))
    | ((df["location_name"] == "Suva") & (df["wind_kph"] == 172.1))
)

df = df[~bad_wind].copy()

In [ ]:
df[["wind_kph", "gust_kph"]].describe()

#### Wind and Gust

Five observations contained implausibly high wind or gust speeds that were inconsistent with the surrounding weather conditions. These records were removed.



---



In [ ]:
df["pressure_mb"].describe()

In [ ]:
df.nlargest(10, "pressure_mb")[
    ["country", "location_name", "last_updated",
     "pressure_mb", "condition_text"]
]

In [ ]:
df.nsmallest(10, "pressure_mb")[
    ["country", "location_name", "last_updated",
     "pressure_mb", "condition_text"]
]

In [ ]:
df[df["location_name"].isin(["Tegucigalpa", "Tehran"])].groupby(
    "location_name"
)["pressure_mb"].agg(["min", "median", "max"])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x="pressure_mb", bins=60, ax=ax[0])
ax[0].set_title("Pressure Distribution")
ax[0].set_xlabel("Pressure (mb)")

sns.boxplot(data=df, x="pressure_mb", ax=ax[1])
ax[1].set_title("Pressure Boxplot")
ax[1].set_xlabel("Pressure (mb)")

plt.tight_layout()
plt.show()

In [ ]:
df = df[
    ~(
        ((df["location_name"] == "Tegucigalpa") & (df["pressure_mb"] == 3006))
        | ((df["location_name"] == "Tehran") & (df["pressure_mb"] == 3000))
    )].copy()

In [ ]:
df["pressure_mb"].describe()

#### Atmospheric Pressure

Two clearly invalid readings of **3000 mb and 3006 mb** were removed. Low-pressure observations at high-elevation locations were retained because they were physically plausible.



---



In [ ]:
df["precip_mm"].describe()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x="precip_mm", bins=60, ax=ax[0])
ax[0].set_title("Precipitation Distribution")
ax[0].set_xlabel("Precipitation (mm)")

sns.boxplot(data=df, x="precip_mm", ax=ax[1])
ax[1].set_title("Precipitation Boxplot")
ax[1].set_xlabel("Precipitation (mm)")

plt.tight_layout()
plt.show()

In [ ]:
df.nlargest(10, "precip_mm")[
    [
        "country",
        "location_name",
        "last_updated",
        "precip_mm",
        "humidity",
        "condition_text"
    ]
]

#### Precipitation

Precipitation is highly right-skewed, so many valid rainy observations were flagged by the IQR method. The highest values were associated with humid or rainy conditions and were retained as plausible extremes.



---



In [ ]:
df.nsmallest(10, "humidity")[
    [
        "country",
        "location_name",
        "last_updated",
        "temperature_celsius",
        "humidity",
        "condition_text"
    ]
]

#### Humidity

Humidity ranged from 2% to 100%. The lowest readings occurred mainly under hot and dry conditions and remained within valid physical bounds.

In [ ]:
df[["visibility_km", "uv_index", "cloud", "wind_degree"]].agg(
    ["min", "median", "max"]
)

Visibility, UV index, cloud cover, and wind direction were also checked for invalid ranges. No additional data-quality issues were found.



---



---



### Air Quality

In [ ]:
aq_cols = [
    "air_quality_Carbon_Monoxide",
    "air_quality_Ozone",
    "air_quality_Nitrogen_dioxide",
    "air_quality_Sulphur_dioxide",
    "air_quality_PM2.5",
    "air_quality_PM10"
]

df[aq_cols].describe().T

In [ ]:
df[df["air_quality_PM10"] < 0][
    [
        "country",
        "location_name",
        "last_updated",
        "air_quality_PM10",
        "air_quality_PM2.5",
        "air_quality_Carbon_Monoxide",
        "air_quality_Ozone",
        "air_quality_Nitrogen_dioxide",
        "air_quality_Sulphur_dioxide"
    ]
]

In [ ]:
df = df[df["air_quality_PM10"] >= 0].copy()

In [ ]:
for col in aq_cols:
    idx = df[col].idxmax()

    print("\n", col)
    print(
        df.loc[idx, [
            "country",
            "location_name",
            "last_updated",
            col,
            "air_quality_PM2.5",
            "air_quality_PM10",
            "air_quality_us-epa-index",
            "air_quality_gb-defra-index"
        ]]
    )

In [ ]:
extreme_locations = [
    "Jakarta",
    "Manama",
    "Kuala Lumpur",
    "Cairo",
    "Santiago",
    "Riyadh"
]

df[df["location_name"].isin(extreme_locations)].groupby("location_name")[aq_cols].agg(
    ["median", "max"]).round(2)

Two observations contained invalid negative PM10 values and were removed. Very high pollutant concentrations were investigated but were supported by other pollution indicators and were retained as plausible extreme events.

In [ ]:
other_cols = [
    "air_quality_us-epa-index",
    "air_quality_gb-defra-index",
    "moon_illumination"
]

df[other_cols].agg(["min", "max"])

### Date-Specific Temperature Anomaly

While checking the data for next-day forecasting, several unusually large temperature changes were found around **17 August 2026**. Since the same pattern appeared across many unrelated locations, the raw observations around this date were checked for a possible batch error.

In [ ]:
(df[
        df["location_name"].isin(
            [
                "Abaer",
                "Bishkek",
                "Harare",
                "Antananarivo",
                "Astana",
                "Tashkent",
                "Baghdad",
                "Riyadh"
            ]
        )
        & df["last_updated"].between("2026-08-14", "2026-08-20")
    ]
    [["country", "location_name", "last_updated", "temperature_celsius"]]
    .sort_values(["location_name", "last_updated"])
)

The suspicious values often appeared as a one-day spike or drop followed by an immediate return to the previous temperature range. To check whether this was isolated or part of a wider issue, the same pattern was searched across the full dataset.

In [ ]:
df.sort_values(
    ["country", "location_name", "last_updated_utc"],
    inplace=True
)

In [ ]:
(
    df.assign(
        previous_temp=lambda x: x.groupby(
            ["country", "location_name"]
        )["temperature_celsius"].shift(1),

        next_temp=lambda x: x.groupby(
            ["country", "location_name"]
        )["temperature_celsius"].shift(-1)
    )
    .loc[
        lambda x:
            ((x["temperature_celsius"] - x["previous_temp"]).abs() > 10)
            & ((x["temperature_celsius"] - x["next_temp"]).abs() > 10)
    ]
    ["last_updated"]
    .dt.date
    .value_counts()
    .head(20)
)

The pattern was strongly concentrated on **17 August 2026**, where 76 observations showed isolated temperature jumps greater than 10°C relative to both neighboring observations. Other dates contained only one or two similar cases.

Because the issue affected many unrelated locations on the same date, these 76 values were treated as a likely corrupted batch rather than normal weather variation.

In [ ]:
bad_aug17 = (
    df.assign(
        previous_temp=lambda x: x.groupby(
            ["country", "location_name"]
        )["temperature_celsius"].shift(1),

        next_temp=lambda x: x.groupby(
            ["country", "location_name"]
        )["temperature_celsius"].shift(-1)
    )
    .loc[
        lambda x:
            (x["last_updated"].dt.date == pd.Timestamp("2026-08-17").date())
            & ((x["temperature_celsius"] - x["previous_temp"]).abs() > 10)
            & ((x["temperature_celsius"] - x["next_temp"]).abs() > 10)
    ]
    .index
)

print("Affected observations:", len(bad_aug17))

Rather than deleting the full observations, only the corrupted temperature values were replaced. This preserves the remaining weather information in those rows.

In [ ]:
df.loc[bad_aug17, "temperature_celsius"] = np.nan

In [ ]:
df.loc[bad_aug17, "temperature_celsius"] = (
    df.groupby(["country", "location_name"])["temperature_celsius"]
      .transform(lambda x: x.interpolate())
      .loc[bad_aug17])

In [ ]:
df.loc[
    bad_aug17,
    [
        "country",
        "location_name",
        "last_updated",
        "temperature_celsius"
    ]
].head(10)

The corrected temperatures now follow the surrounding local pattern. Only the identified August 17 values were interpolated; isolated large temperature changes on other dates were retained unless there was additional evidence that they were invalid.

### Final Data Quality Check

In [ ]:
df.isnull().sum()[df.isnull().sum() > 0]

In [ ]:
object_cols = df.select_dtypes(include="O").columns

for col in object_cols:
    empty_count = df[col].str.strip().eq("").sum()
    if empty_count > 0:
        print(col, ":", empty_count)

In [ ]:
df.dropna(inplace=True)

In [ ]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Final shape:", df.shape)

After cleaning, the dataset contains no remaining missing values or duplicate rows. Clear data errors were corrected or removed, while plausible extreme weather and air-quality observations were retained.

## 3. Exploratory Data Analysis

### 3.1 Dataset Coverage

The dataset covers multiple countries and locations, but observation history is not necessarily equal across them. Coverage is checked before comparing long-term patterns.

**Question:** Is the observation history consistent across locations, and is the dataset geographically balanced?

In [ ]:
print("Date range:", df["last_updated"].min(), "to", df["last_updated"].max())
print("Countries:", df["country"].nunique())
print("Locations:", df["location_name"].nunique())
print("Observations:", len(df))

In [ ]:
records_per_location = (
    df.groupby(["country", "location_name"])
      .size()
      .describe()
)

records_per_location

In [ ]:
location_counts = (
    df.groupby(["country", "location_name"])
      .size()
      .sort_values()
)

plt.figure(figsize=(10, 5))
sns.histplot(location_counts, bins=30)

plt.title("Number of Observations per Location")
plt.xlabel("Number of Observations")
plt.ylabel("Number of Locations")

plt.show()

In [ ]:
location_coverage = (
    df.groupby(["country", "location_name"])
      .agg(
          first_observation=("last_updated", "min"),
          last_observation=("last_updated", "max"),
          observations=("last_updated", "size")
      )
      .sort_values("first_observation")
)

location_coverage.head()

In [ ]:
location_coverage.tail()

Coverage is uneven across locations. Most locations have long observation histories, while some entered the dataset later. For seasonal comparisons, a well-covered subset is used so that incomplete histories do not distort the patterns.

For seasonal EDA, locations with at least 800 observations are treated as well-covered. This threshold keeps locations with close to the full observation history while excluding locations that entered the dataset much later.

This filter is only used for EDA comparisons, not as a rule for model training.

In [ ]:
# Identify locations with long observation histories
location_counts = (
    df.groupby(["country", "location_name"])
      .size()
)

print("Locations with >= 800 observations:", (location_counts >= 800).sum())
print("Total locations:", len(location_counts))
print("Percentage:", round((location_counts >= 800).mean() * 100, 1), "%")

In [ ]:
well_covered = location_counts[location_counts >= 800].index

df_covered = (
    df.set_index(["country", "location_name"])
      .loc[well_covered]
      .reset_index()
)

print(df_covered.shape)
print(df_covered["location_name"].nunique())

In [ ]:
location_info = (
    df.groupby(["country", "location_name"])
      .agg(
          latitude=("latitude", "median"),
          longitude=("longitude", "median"),
          observations=("last_updated", "size"),
          first_observation=("last_updated", "min"),
          last_observation=("last_updated", "max")
      )
      .reset_index()
)

location_info["coverage"] = np.where(
    location_info["observations"] >= 800,
    "Well-covered",
    "Limited coverage"
)

location_info["hemisphere"] = np.where(
    location_info["latitude"] >= 0,
    "Northern",
    "Southern"
)

In [ ]:
pd.crosstab(
    location_info["hemisphere"],
    location_info["coverage"],
    margins=True
)

**Finding:** Coverage is uneven across locations, with some locations entering the dataset later than others. The data is also weighted toward the Northern Hemisphere. For seasonal comparisons, well-covered locations are used to reduce the effect of incomplete histories, while keeping this geographic imbalance in mind.

### 3.2 Temperature Patterns

Temperature is examined over time using the well-covered locations to reduce the effect of unequal observation histories.

**Question:** How does temperature change over the observation period, and how does the pattern differ geographically?

In [ ]:
df_covered["month"] = df_covered["last_updated"].dt.to_period("M").dt.to_timestamp()

monthly_temp = (
    df_covered.groupby("month")["temperature_celsius"]
              .mean()
              .reset_index()
)
plt.figure(figsize=(12, 5))

sns.lineplot(
    data=monthly_temp,
    x="month",
    y="temperature_celsius",
    marker="o"
)

plt.title("Average Monthly Temperature Across Well-Covered Locations")
plt.xlabel("Month")
plt.ylabel("Average Temperature (°C)")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

**Finding:** Average temperature follows a repeating seasonal cycle rather than a steady long-term increase or decrease. Since Northern Hemisphere locations are more heavily represented, the overall pattern is influenced by Northern seasonality.

In [ ]:
sns.lineplot(
    data=(
        df_covered
        .assign(
            hemisphere=np.where(
                df_covered["latitude"] >= 0,
                "Northern",
                "Southern"
            ),
            month=df_covered["last_updated"].dt.month
        )
        .groupby(["hemisphere", "month"], as_index=False)["temperature_celsius"]
        .mean()
    ),
    x="month",
    y="temperature_celsius",
    hue="hemisphere",
    marker="o"
)

plt.title("Average Monthly Temperature by Hemisphere")
plt.xlabel("Month")
plt.ylabel("Average Temperature (°C)")
plt.xticks(range(1, 13))
plt.show()

**Finding:** The seasonal cycles are opposite across hemispheres. Northern locations are warmest around June–August, while Southern locations are warmest around December–February. This shows why global averages can hide geographic differences.

### 3.3 Precipitation Patterns

**Question:** Does precipitation show seasonal patterns, and do those patterns differ geographically?

Precipitation is recorded at the observation level rather than as a daily total. Since it contains many zero values and is strongly right-skewed, frequency and typical precipitation amount are analyzed separately.

In [ ]:
print("Skewness:", df["precip_mm"].skew())
print(
    "Percentage of observations with no precipitation:",
    round((df["precip_mm"] == 0).mean() * 100, 2),
    "%"
)

**Finding:** Most observations are dry, and a small number of large precipitation values create a strongly right-skewed distribution. For this reason, median precipitation is more representative of a typical wet observation than the mean.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Overall precipitation frequency
sns.lineplot(
    data=(
        df_covered
        .assign(month=df_covered["last_updated"].dt.month)
        .groupby("month")["precip_mm"]
        .apply(lambda x: (x > 0).mean() * 100)
        .reset_index(name="frequency")
    ),
    x="month",
    y="frequency",
    marker="o",
    ax=ax[0]
)

# Precipitation frequency by hemisphere
sns.lineplot(
    data=(
        df_covered
        .assign(
            month=df_covered["last_updated"].dt.month,
            hemisphere=np.where(
                df_covered["latitude"] >= 0,
                "Northern",
                "Southern"
            )
        )
        .groupby(["hemisphere", "month"])["precip_mm"]
        .apply(lambda x: (x > 0).mean() * 100)
        .reset_index(name="frequency")
    ),
    x="month",
    y="frequency",
    hue="hemisphere",
    marker="o",
    ax=ax[1]
)

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

ax[0].set_title("Overall Precipitation Frequency")
ax[1].set_title("Precipitation Frequency by Hemisphere")

for a in ax:
    a.set_xlabel("Month")
    a.set_ylabel("Observations with Precipitation (%)")
    a.set_xticks(range(1, 13))
    a.set_xticklabels(months)

plt.tight_layout()
plt.show()

**Finding:** Overall precipitation frequency changes only slightly across months, but the hemisphere comparison reveals clearer seasonal differences, particularly in Southern Hemisphere locations.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Overall precipitation amount
sns.lineplot(
    data=(
        df_covered[df_covered["precip_mm"] > 0]
        .assign(month=lambda x: x["last_updated"].dt.month)
        .groupby("month", as_index=False)["precip_mm"]
        .median()
    ),
    x="month",
    y="precip_mm",
    marker="o",
    ax=ax[0]
)

# Precipitation amount by hemisphere
sns.lineplot(
    data=(
        df_covered[df_covered["precip_mm"] > 0]
        .assign(
            month=lambda x: x["last_updated"].dt.month,
            hemisphere=lambda x: np.where(
                x["latitude"] >= 0, "Northern", "Southern"
            )
        )
        .groupby(["hemisphere", "month"], as_index=False)["precip_mm"]
        .median()
    ),
    x="month",
    y="precip_mm",
    hue="hemisphere",
    marker="o",
    ax=ax[1]
)

ax[0].set_title("Typical Precipitation Amount")
ax[1].set_title("Typical Precipitation Amount by Hemisphere")

for a in ax:
    a.set_xlabel("Month")
    a.set_ylabel("Median Precipitation (mm)")
    a.set_xticks(range(1, 13))
    a.set_xticklabels(months)

plt.tight_layout()
plt.show()

**Finding:** Precipitation amount also shows stronger seasonal variation after separating the hemispheres. As with temperature, global aggregation can hide meaningful regional weather patterns.

### 3.4 Weather Relationships

**Question:** How are the main weather variables related to each other, and do Pearson and Spearman correlations tell the same story?

In [ ]:
weather_cols = [
    "temperature_celsius",
    "feels_like_celsius",
    "humidity",
    "pressure_mb",
    "wind_kph",
    "gust_kph",
    "precip_mm",
    "visibility_km",
    "cloud",
    "uv_index"
]

plt.figure(figsize=(11, 8))

sns.heatmap(
    df[weather_cols].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Between Weather Variables")
plt.tight_layout()
plt.show()

In [ ]:
df[weather_cols].corr(method="spearman").round(2)

**Finding:** Temperature and feels-like temperature have the strongest relationship (`r ≈ 0.98`), while wind speed and gust speed are also strongly related (`r ≈ 0.93`). Temperature also shows a moderate positive relationship with UV index, while humidity is positively related to cloud cover and negatively related to UV.

Pearson and Spearman give similar results for most weather variables. The main difference is precipitation, where Spearman shows stronger relationships with variables such as cloud cover and humidity. This suggests that some precipitation relationships are monotonic rather than strictly linear.



---



### 3.5 Air Quality and Weather

**Question:** Which weather variables have the strongest relationships with air pollutants?

In [ ]:
aq_cols = [
    "air_quality_Carbon_Monoxide",
    "air_quality_Ozone",
    "air_quality_Nitrogen_dioxide",
    "air_quality_Sulphur_dioxide",
    "air_quality_PM2.5",
    "air_quality_PM10"
]

In [ ]:
plt.figure(figsize=(10, 7))

sns.heatmap(
    df[weather_cols + aq_cols]
    .corr()
    .loc[weather_cols, aq_cols],
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Between Weather Conditions and Air Pollutants")
plt.xlabel("Air Pollutants")
plt.ylabel("Weather Variables")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))

sns.heatmap(
    df[weather_cols + aq_cols]
    .corr(method="spearman")
    .loc[weather_cols, aq_cols],
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Spearman Correlation Between Weather Conditions and Air Pollutants")
plt.xlabel("Air Pollutants")
plt.ylabel("Weather Variables")

plt.tight_layout()
plt.show()

**Finding:** Pearson and Spearman correlations show similar directions for several weather–pollution relationships, but Spearman reveals stronger associations for some skewed variables.

Ozone tends to be higher with higher UV and lower humidity. Stronger wind and gusts are generally associated with lower CO and NO₂, while precipitation shows a clearer negative relationship with particulate matter under Spearman correlation.

Because several pollutant and precipitation variables are highly skewed, these relationships are interpreted as associations rather than causal effects.

**Question:** Are PM2.5 and PM10 concentrations different when precipitation is observed?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(
    data=df,
    x=df["precip_mm"] > 0,
    y="air_quality_PM2.5",
    showfliers=False,
    ax=ax[0]
)

sns.boxplot(
    data=df,
    x=df["precip_mm"] > 0,
    y="air_quality_PM10",
    showfliers=False,
    ax=ax[1]
)

ax[0].set_title("PM2.5 by Precipitation Presence")
ax[1].set_title("PM10 by Precipitation Presence")

for a in ax:
    a.set_xlabel("Precipitation Observed")
    a.set_xticks([0, 1])
    a.set_xticklabels(["No", "Yes"])

ax[0].set_ylabel("PM2.5")
ax[1].set_ylabel("PM10")

plt.tight_layout()
plt.show()

**Finding:** PM2.5 and PM10 are generally lower in observations where precipitation is recorded. This supports the negative Spearman relationship between precipitation and particulate matter, although it does not establish causation.

**Question:** Do particulate pollution levels also show seasonal differences between hemispheres?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

for pollutant, axis, title in [
    ("air_quality_PM2.5", ax[0], "PM2.5"),
    ("air_quality_PM10", ax[1], "PM10")
]:
    sns.lineplot(
        data=(
            df_covered
            .assign(
                month=lambda x: x["last_updated"].dt.month,
                hemisphere=lambda x: np.where(
                    x["latitude"] >= 0, "Northern", "Southern"
                )
            )
            .groupby(["hemisphere", "month"], as_index=False)[pollutant]
            .median()
        ),
        x="month",
        y=pollutant,
        hue="hemisphere",
        marker="o",
        ax=axis
    )

    axis.set_title(f"Monthly {title} by Hemisphere")
    axis.set_xlabel("Month")
    axis.set_ylabel(f"Median {title}")
    axis.set_xticks(range(1, 13))
    axis.set_xticklabels(months)

plt.tight_layout()
plt.show()

**Finding:** Particulate pollution shows different seasonal patterns across hemispheres. Northern Hemisphere locations generally have higher median PM2.5 and PM10 during the cooler part of the year, while Southern Hemisphere locations follow a different seasonal cycle.

These patterns may reflect several seasonal and geographic factors, not precipitation alone.



---



**Question:** How does ozone vary with humidity and UV index?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Ozone across humidity ranges
sns.lineplot(
    data=(
        df.assign(
            humidity_bin=pd.cut(df["humidity"], bins=10)
        )
        .groupby("humidity_bin", observed=True)
        .agg(
            humidity=("humidity", "median"),
            ozone=("air_quality_Ozone", "median")
        )
        .reset_index()
    ),
    x="humidity",
    y="ozone",
    marker="o",
    ax=ax[0]
)

# Ozone across UV ranges
sns.lineplot(
    data=(
        df.assign(
            uv_bin=pd.cut(df["uv_index"], bins=10)
        )
        .groupby("uv_bin", observed=True)
        .agg(
            uv=("uv_index", "median"),
            ozone=("air_quality_Ozone", "median")
        )
        .reset_index()
    ),
    x="uv",
    y="ozone",
    marker="o",
    ax=ax[1]
)

ax[0].set_title("Ozone Levels Across Humidity")
ax[0].set_xlabel("Humidity (%)")
ax[0].set_ylabel("Median Ozone")

ax[1].set_title("Ozone Levels Across UV Index")
ax[1].set_xlabel("UV Index")
ax[1].set_ylabel("Median Ozone")

plt.tight_layout()
plt.show()

**Finding:** Median ozone generally decreases as humidity increases and rises across higher UV levels. The UV relationship is not perfectly linear, but both patterns are consistent with the correlation analysis.



---



**Question:** Does ozone also show seasonal differences between hemispheres?

In [ ]:
sns.lineplot(
    data=(
        df_covered
        .assign(
            month=lambda x: x["last_updated"].dt.month,
            hemisphere=lambda x: np.where(
                x["latitude"] >= 0, "Northern", "Southern"
            )
        )
        .groupby(["hemisphere", "month"], as_index=False)["air_quality_Ozone"]
        .median()
    ),
    x="month",
    y="air_quality_Ozone",
    hue="hemisphere",
    marker="o"
)

plt.title("Monthly Ozone by Hemisphere")
plt.xlabel("Month")
plt.ylabel("Median Ozone")
plt.xticks(range(1, 13), months)

plt.tight_layout()
plt.show()

**Finding:** Ozone shows different seasonal cycles across hemispheres. Northern Hemisphere locations tend to have higher median ozone around the middle of the year, while Southern Hemisphere locations follow a different seasonal pattern.

This suggests that ozone levels are influenced by both atmospheric conditions and geography.

#### Air Quality Summary

Weather and air quality show several consistent associations. Precipitation is linked with lower particulate pollution, while ozone is related to humidity and UV conditions. Pollution levels also vary seasonally across hemispheres, showing that both weather and geography matter when interpreting air-quality patterns.



---



---



### 3.6 Geographic and Temporal Patterns

**Question:** How does typical temperature vary with latitude?

In [ ]:
median_temp = (
    df.groupby(["country", "location_name"])["temperature_celsius"]
      .median()
      .rename("median_temperature")
)

location_info = (
    location_info
    .drop(columns="median_temperature", errors="ignore")
    .join(
        median_temp,
        on=["country", "location_name"]
    ))

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=location_info,
    x="latitude",
    y="median_temperature",
    hue="coverage",
    alpha=0.7
)

plt.title("Typical Temperature Across Latitudes")
plt.xlabel("Latitude")
plt.ylabel("Median Temperature (°C)")

plt.axvline(0, linestyle="--", linewidth=1)

plt.tight_layout()
plt.show()

**Finding:** Typical temperature generally decreases as locations move farther from the equator, although the relationship is not perfectly linear.

Limited-coverage locations are included to show the full geographic spread, but their medians should be interpreted cautiously because some locations may represent only part of the seasonal cycle.



---



**Question:** How is typical temperature distributed across the sampled locations worldwide?

In [ ]:
fig = px.scatter_geo(
    location_info,
    lat="latitude",
    lon="longitude",
    color="median_temperature",
    hover_name="location_name",
    hover_data={
        "country": True,
        "median_temperature": ":.1f",
        "observations": True,
        "coverage": True,
        "latitude": False,
        "longitude": False
    },
    color_continuous_scale="RdYlBu_r",
    projection="natural earth",
    title="Geographical Distribution of Typical Temperature"
)

fig.update_traces(marker=dict(size=7, opacity=0.8))

fig.update_layout(
    coloraxis_colorbar_title="Median<br>Temperature (°C)"
)

fig.show()

**Finding:** Warmer sampled locations are concentrated mainly in tropical and subtropical regions, while cooler locations are more common at higher latitudes. Locations at similar latitudes can still have different median temperatures, showing that geographic position alone does not capture all of the observed temperature variation.



---



**Question:** How representative are the sampled locations at the country level?

In [ ]:
location_info.groupby("country").size().sort_values(ascending=False).head(20)

In [ ]:
(location_info.groupby("country").size() == 1).sum()

**Finding:** Geographic representation within countries is limited. Of the 191 standardized country labels, 149 are represented by only one sampled location.

For this reason, country-level averages should not be interpreted as nationally representative. Spatial conclusions are better made at the location, latitude, or hemisphere level.



---



**Question:** What does the full observation timeline show about temperature variation across hemispheres?

In [ ]:
(
    df_covered
    .assign(
        hemisphere=np.where(
            df_covered["latitude"] >= 0,
            "Northern",
            "Southern"
        )
    )
    .set_index("last_updated")
    .groupby("hemisphere")["temperature_celsius"]
    .resample("MS")
    .mean()
    .unstack(0)
    .plot(
        figsize=(12, 5),
        marker="o",
        title="Monthly Temperature Variation by Hemisphere"
    )
)

plt.xlabel("Date")
plt.ylabel("Average Temperature (°C)")
plt.show()

**Finding:** The timeline shows repeating seasonal cycles in both hemispheres, with opposite timing and stronger variation in the Northern Hemisphere.

The dataset covers only about May 2024 to September 2026, with incomplete coverage at the beginning and end. This is enough to examine seasonality and short-term variation, but not to establish long-term climate trends or climate change.

### EDA Summary

The analysis showed strong seasonal and geographic structure in the weather data. Temperature follows clear seasonal cycles that differ by hemisphere, while precipitation patterns become more informative when frequency and amount are analyzed separately.

Several weather variables are strongly related, particularly temperature with feels-like temperature and wind with gust speed. Weather conditions also show associations with air quality, including lower particulate pollution during precipitation and relationships between ozone, humidity, and UV.

The dataset is geographically and temporally unbalanced, so location coverage must be considered when interpreting aggregated patterns. These findings support using time, location, seasonality, and recent temperature information when building the forecasting model.



---



---



## 4. Forecasting Setup and Feature Engineering

Before modeling, the observation timing is checked and the next-day forecasting target is defined carefully so lag features and training examples reflect real elapsed time.

### 4.1 Temporal Structure
Before creating lag features, the observation spacing was checked to confirm whether records were consistently close to 24 hours apart.

In [ ]:
# Are the observations acctually daily for each location
(
    df.sort_values(["country", "location_name", "last_updated_utc"])
      .groupby(["country", "location_name"])["last_updated_utc"]
      .diff()
      .describe()
)

Some locations contain irregular gaps between observations. The gap distribution is checked to see how often the previous row actually represents roughly one day earlier.

In [ ]:
gaps = (
    df.sort_values(["country", "location_name", "last_updated_utc"])
      .groupby(["country", "location_name"])["last_updated_utc"]
      .diff()
      .dt.total_seconds()
      .div(3600)
)

print("Gaps between 20 and 28 hours:", gaps.between(20, 28).sum())
print("Percentage:", round(gaps.between(20, 28).mean() * 100, 2), "%")

In [ ]:
gaps[~gaps.between(20, 28) & gaps.notna()].describe()

### Forecasting Design

The task is to predict temperature approximately 24 hours ahead for the same location using only information available at forecast time.

A global model is used across locations, without excluding shorter-history locations. Time gaps are handled explicitly, and missing recent history is retained so fallback behavior can be supported later. Model evaluation is time-based rather than random.

A small number of locations contain more than one observation on the same calendar day, so these cases are checked before deciding whether records can be collapsed by date.

In [ ]:
(
    df.assign(date=df["last_updated"].dt.date)
      .groupby(["country", "location_name", "date"])
      .size()
      .value_counts()
      .sort_index()
)

In [ ]:
#investigating locatiosn with multible observations a day
(
    df.assign(date=df["last_updated"].dt.date)
      .groupby(["country", "location_name", "date"])
      .filter(lambda x: len(x) > 1)
      [["country", "location_name", "last_updated", "temperature_celsius"]]
      .sort_values(["country", "location_name", "last_updated"])
      .head(30)
)

Some same-day observations are separated by nearly 24 hours. Collapsing records only by calendar date could therefore merge observations from different forecasting intervals, so elapsed time is used instead.



---



### 4.2 Forecast Target

The forecasting target is the temperature at the next observation approximately 24 hours later for the same location.

In [ ]:
model_df = df.sort_values(
    ["country", "location_name", "last_updated_utc"]
).copy()

model_df["target_temperature"] = (
    model_df.groupby(["country", "location_name"])["temperature_celsius"]
            .shift(-1)
)

model_df["target_time"] = (
    model_df.groupby(["country", "location_name"])["last_updated_utc"]
            .shift(-1)
)

model_df["hours_to_target"] = (
    model_df["target_time"]
    - model_df["last_updated_utc"]
).dt.total_seconds() / 3600

In [ ]:
model_df["hours_to_target"].describe()

Several windows around 24 hours are compared to choose a practical next-day horizon without discarding unnecessary training examples.

In [ ]:
for lower, upper in [(23, 25), (22, 26), (20, 28)]:
    print(
        f"{lower}-{upper} hours:",
        model_df["hours_to_target"].between(lower, upper).sum()
    )

The 22–26 hour window is used because it recovers substantially more examples than 23–25 hours, while widening further to 20–28 hours adds very few additional observations.

In [ ]:
model_df = model_df[
    model_df["hours_to_target"].between(22, 26)
].copy()

### 4.3 Feature Engineering

The initial features include current weather conditions, seasonal information, geographic context, and recent temperature history. Features are created only from information available at or before the forecast time.

Highly redundant variables identified during EDA, such as `feels_like_celsius` and `gust_kph`, are excluded from the candidate set.

In [ ]:
model_df["month"] = model_df["last_updated"].dt.month
model_df["day_of_year"] = model_df["last_updated"].dt.dayofyear

In [ ]:
(model_df["target_temperature"] - model_df["temperature_celsius"]).abs().describe()

Most next-day temperature changes are relatively small, which suggests that current temperature should provide a strong persistence baseline.

In [ ]:
df.sort_values(
    ["country", "location_name", "last_updated_utc"],
    inplace=True
)

In [ ]:
df["lag_1"] = (
    df.groupby(["country", "location_name"])["temperature_celsius"]
      .shift(1)
)

In [ ]:
df["lag_1"] = df["lag_1"].where(
    df.groupby(["country", "location_name"])["last_updated_utc"]
      .diff()
      .dt.total_seconds()
      .div(3600)
      .between(22, 26)
)

In [ ]:
model_df["lag_1"] = df.loc[model_df.index, "lag_1"]

In [ ]:
model_df[
    ["location_name", "last_updated",
     "lag_1", "temperature_celsius", "target_temperature"]
].head(10)

In [ ]:
print("Missing lag_1:", model_df["lag_1"].isna().sum())
print( "Percentage:",round(model_df["lag_1"].isna().mean() * 100, 2),"%")

`lag_1` is missing for about 0.93% of valid forecasting examples. These rows are retained so the pipeline can still support cases where recent history is unavailable.

In [ ]:
candidate_features = [
    "temperature_celsius",
    "lag_1",

    "humidity",
    "pressure_mb",
    "wind_kph",
    "precip_mm",
    "visibility_km",
    "cloud",
    "uv_index",

    "month",
    "day_of_year",

    "latitude",
    "longitude",
    "country",
    "location_name"
]

In [ ]:
model_columns = [
    "last_updated",
    "last_updated_utc",
    "target_time",
    "hours_to_target",
    "target_temperature"
] + candidate_features

model_data = model_df[model_columns].copy()

In [ ]:
model_data.info()

In [ ]:
print("Contries: ", model_data['country'].nunique())
print(
    "Locations:",
    model_data[["country", "location_name"]]
      .drop_duplicates()
      .shape[0]
)

In [ ]:
model_data.describe()

Month and day of year are also represented cyclically so the model can capture seasonal continuity, such as December being close to January.

In [ ]:
#cyclic features
model_data["month_sin"] = np.sin(2 * np.pi * model_data["month"] / 12)
model_data["month_cos"] = np.cos(2 * np.pi * model_data["month"] / 12)

model_data["day_of_year_sin"] = np.sin(
    2 * np.pi * model_data["day_of_year"] / 365.25
)
model_data["day_of_year_cos"] = np.cos(
    2 * np.pi * model_data["day_of_year"] / 365.25
)

Because precipitation is strongly right-skewed, a log-transformed version is created using `log1p`.

In [ ]:
# extreamly right skewed
model_data["log_precip_mm"] = np.log1p(model_data["precip_mm"])

In [ ]:
# model_data["lag_1_missing"] = model_data["lag_1"].isna().astype(int)

#### Additional Temperature History

Initial modeling showed that recent temperature history contained the strongest predictive signal. Additional lags were therefore created using actual elapsed time rather than row position alone.

In [ ]:
for lag in [2, 3, 7]:
    previous_temp = (
        df.groupby(["country", "location_name"])["temperature_celsius"]
          .shift(lag))

    previous_time = (
        df.groupby(["country", "location_name"])["last_updated_utc"]
          .shift(lag))

    hours_back = (
        df["last_updated_utc"] - previous_time
        ).dt.total_seconds() / 3600

    expected_hours = lag * 24

    df[f"lag_{lag}"] = previous_temp.where(
        hours_back.between(
            expected_hours - 4,
            expected_hours + 4))

In [ ]:
for feature in ["lag_2", "lag_3", "lag_7"]:
    model_data[feature] = df.loc[model_data.index, feature]

In [ ]:
model_data[
    ["lag_1", "lag_2", "lag_3", "lag_7"]].isna().mean().mul(100).round(2)

In [ ]:
model_data["temp_change_1d"] = (
    model_data["temperature_celsius"]
    - model_data["lag_1"]
)

In [ ]:
model_data["temp_mean_3d"] = (
    model_data[
        ["temperature_celsius", "lag_1", "lag_2"]
    ]
    .mean(axis=1)
)

recent_3_count = model_data[
    ["temperature_celsius", "lag_1", "lag_2"]
].notna().sum(axis=1)

model_data.loc[
    recent_3_count < 2,
    "temp_mean_3d"
] = np.nan

The additional history features remain mostly available across the dataset. Missing lag values are retained rather than filled globally, allowing each model to handle them appropriately during preprocessing.

#### Feature Engineering Summary

The final features combine:

- current temperature and recent temperature history
- current weather conditions
- cyclical seasonal features
- geographic information
- a log transformation of precipitation.

Temperature lags are retained only when their elapsed time is consistent with the intended historical interval. Missing lag values are kept and handled according to each model's preprocessing requirements.

## 5. Modeling and Evaluation

### 5.1 Temporal Split

The data is split chronologically into training, validation, and test sets. This prevents future observations from being used to train the models and keeps the evaluation consistent with the forecasting task.

In [ ]:
def evaluate_model(y_true, y_pred, dataset_name):
    print(dataset_name)
    print("MAE:", mean_absolute_error(y_true, y_pred))
    print("RMSE:", mean_squared_error(y_true, y_pred) ** 0.5)
    print("R²:", r2_score(y_true, y_pred))
    print()

In [ ]:
train_cutoff = model_data["last_updated_utc"].quantile(0.70)
val_cutoff = model_data["last_updated_utc"].quantile(0.85)

In [ ]:
train_df = model_data[
    model_data["target_time"] < train_cutoff
].copy()

val_df = model_data[
    (model_data["last_updated_utc"] >= train_cutoff)
    & (model_data["target_time"] < val_cutoff)
].copy()

test_df = model_data[
    model_data["last_updated_utc"] >= val_cutoff
].copy()

In [ ]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print(
    "Train labels end before validation forecasts begin:",
    train_df["target_time"].max()
    < val_df["last_updated_utc"].min()
)

print(
    "Validation labels end before test forecasts begin:",
    val_df["target_time"].max()
    < test_df["last_updated_utc"].min()
)

### 5.2 Persistence Baseline

The persistence baseline assumes that the next day's temperature will be the same as the current temperature. It provides a simple reference point for evaluating the forecasting models.

In [ ]:
baseline_train = train_df["temperature_celsius"]
baseline_val = val_df["temperature_celsius"]
baseline_test = test_df["temperature_celsius"]

evaluate_model(train_df["target_temperature"],baseline_train,"Train")
evaluate_model(val_df["target_temperature"],baseline_val,"Validation")
evaluate_model(test_df["target_temperature"],baseline_test,"Test")

### 5.3 Initial CatBoost Model

CatBoost was selected as the first machine-learning model because it can handle the categorical location features directly and supports missing values in numeric features such as `lag_1`.

In [ ]:
train_df.columns

In [ ]:
catboost_features = [
    # Temperature history
    "temperature_celsius",
    "lag_1",

    # Current weather
    "humidity",
    "pressure_mb",
    "wind_kph",
    "log_precip_mm",
    "visibility_km",
    "cloud",
    "uv_index",

    # Seasonality
    "month",
    "day_of_year",
    "month_sin",
    "month_cos",
    "day_of_year_sin",
    "day_of_year_cos",

    # Geography
    "latitude",
    "longitude",
    "country",
    "location_name"
]

In [ ]:
X_train = train_df[catboost_features]
y_train = train_df["target_temperature"]

X_val = val_df[catboost_features]
y_val = val_df["target_temperature"]

X_test = test_df[catboost_features]
y_test = test_df["target_temperature"]

In [ ]:
catboost_model = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

catboost_model.fit(
    X_train,
    y_train,
    cat_features=["country", "location_name"],
    eval_set=(X_val, y_val),
    early_stopping_rounds=50
)

In [ ]:
catboost_train_pred = catboost_model.predict(X_train)
catboost_val_pred = catboost_model.predict(X_val)
catboost_test_pred = catboost_model.predict(X_test)

evaluate_model(y_train,catboost_train_pred,"Train")
evaluate_model(y_val,catboost_val_pred,"Validation")
evaluate_model(y_test,catboost_test_pred,"Test")

In [ ]:
model_comparison = pd.DataFrame({
    "Model": [
        "Persistence",
        "CatBoostInitial"
    ],
    "Test MAE": [
        mean_absolute_error(y_test, baseline_test),
        mean_absolute_error(y_test, catboost_test_pred),
    ],
    "Test RMSE": [
        mean_squared_error(y_test, baseline_test) ** 0.5,
        mean_squared_error(y_test, catboost_test_pred) ** 0.5
    ],
    "Test R²": [
        r2_score(y_test, baseline_test),
        r2_score(y_test, catboost_test_pred)
    ]
})

model_comparison.sort_values("Test RMSE")

The initial CatBoost model improves on the persistence baseline, but the improvement is relatively small. This suggests that adding more information about recent temperature history may be worth testing.

### 5.4 HistGradientBoosting

HistGradientBoosting is tested as another tree-based model using the numeric features.

In [ ]:
hist_features = [
    # Temperature history
    "temperature_celsius",
    "lag_1",

    # Current weather
    "humidity",
    "pressure_mb",
    "wind_kph",
    "log_precip_mm",
    "visibility_km",
    "cloud",
    "uv_index",

    # Seasonality
    "month_sin",
    "month_cos",
    "day_of_year_sin",
    "day_of_year_cos",

    # Geography
    "latitude",
    "longitude"
]

In [ ]:
X_train_hist = train_df[hist_features]
X_val_hist = val_df[hist_features]
X_test_hist = test_df[hist_features]

y_train_hist = train_df["target_temperature"]
y_val_hist = val_df["target_temperature"]
y_test_hist = test_df["target_temperature"]

In [ ]:
hist_model = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=500,
    max_depth=6,
    random_state=42,
    early_stopping=False
)
hist_model.fit(
    X_train_hist,
    y_train_hist
)

In [ ]:
hist_train_pred = hist_model.predict(X_train_hist)
hist_val_pred = hist_model.predict(X_val_hist)
hist_test_pred = hist_model.predict(X_test_hist)

evaluate_model(y_train_hist,hist_train_pred,"Train")
evaluate_model(y_val_hist,hist_val_pred,"Validation")
evaluate_model(y_test_hist,hist_test_pred,"Test")

In [ ]:
hist_result = pd.DataFrame([{
    "Model": "HistGradientBoosting",
    "Test MAE": mean_absolute_error(y_test_hist, hist_test_pred),
    "Test RMSE": mean_squared_error(y_test_hist, hist_test_pred) ** 0.5,
    "Test R²": r2_score(y_test_hist, hist_test_pred)
}])

model_comparison = pd.concat(
    [model_comparison, hist_result],
    ignore_index=True
)

model_comparison.sort_values("Test RMSE")

### 5.5 CatBoost Hyperparameter Tuning

In [ ]:
param_grid = [
    {"depth": 4, "learning_rate": 0.05, "l2_leaf_reg": 3},
    {"depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 3},
    {"depth": 8, "learning_rate": 0.05, "l2_leaf_reg": 3},
    {"depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 3},
    {"depth": 6, "learning_rate": 0.10, "l2_leaf_reg": 3},
    {"depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 5},
    {"depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 10},
]

In [ ]:
tuning_results = []

for params in param_grid:
    model = CatBoostRegressor(
        iterations=1000,
        depth=params["depth"],
        learning_rate=params["learning_rate"],
        l2_leaf_reg=params["l2_leaf_reg"],
        loss_function="RMSE",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train,
        y_train,
        cat_features=["country", "location_name"],
        eval_set=(X_val, y_val),
        early_stopping_rounds=50
    )

    val_pred = model.predict(X_val)

    tuning_results.append({
        "depth": params["depth"],
        "learning_rate": params["learning_rate"],
        "l2_leaf_reg": params["l2_leaf_reg"],
        "best_iteration": model.get_best_iteration(),
        "val_mae": mean_absolute_error(y_val, val_pred),
        "val_rmse": mean_squared_error(y_val, val_pred) ** 0.5,
        "val_r2": r2_score(y_val, val_pred)
    })

In [ ]:
tuning_df = pd.DataFrame(tuning_results)
tuning_df.sort_values("val_rmse")

The tested CatBoost configurations produced very similar validation performance. The best validation RMSE was obtained with depth 6 and a learning rate of 0.10, so these settings were used for the later CatBoost experiments.



---



---



### 5.6 Ridge Regression
Ridge regression is included as a linear baseline using the same core weather and geographic features. The numeric features are standardized and categorical variables are one-hot encoded.


In [ ]:
categorical_features_ridge = [
    "country",
    "location_name"
]

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, hist_features),
    ("cat", categorical_pipeline, categorical_features_ridge)
])

In [ ]:
ridge_features = (hist_features+ categorical_features_ridge)

X_train_ridge = train_df[ridge_features]
y_train_ridge = train_df["target_temperature"]

X_val_ridge = val_df[ridge_features]
y_val_ridge = val_df["target_temperature"]

X_test_ridge = test_df[ridge_features]
y_test_ridge = test_df["target_temperature"]

In [ ]:
ridge_alphas = [0.01,0.1,1,10,100]

In [ ]:
ridge_results = []

for alpha in ridge_alphas:

    ridge_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", Ridge(
            alpha=alpha,
            solver="lsqr"
        ))
    ])

    ridge_pipeline.fit(
        X_train_ridge,
        y_train_ridge
    )

    train_pred = ridge_pipeline.predict(X_train_ridge)
    val_pred = ridge_pipeline.predict(X_val_ridge)

    ridge_results.append({
        "alpha": alpha,

        "train_mae": mean_absolute_error(
            y_train_ridge, train_pred
        ),
        "train_rmse": mean_squared_error(
            y_train_ridge, train_pred
        ) ** 0.5,
        "train_r2": r2_score(
            y_train_ridge, train_pred
        ),

        "val_mae": mean_absolute_error(
            y_val_ridge, val_pred
        ),
        "val_rmse": mean_squared_error(
            y_val_ridge, val_pred
        ) ** 0.5,
        "val_r2": r2_score(
            y_val_ridge, val_pred
        )
    })

In [ ]:
ridge_results_df = pd.DataFrame(ridge_results)
ridge_results_df.sort_values("val_rmse")

In [ ]:
best_alpha = (
    ridge_results_df
    .sort_values("val_rmse")
    .iloc[0]["alpha"]
)

In [ ]:
ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(
        alpha=best_alpha,
        solver="lsqr"
    ))])
ridge_model.fit(
    X_train_ridge,
    y_train_ridge
)

In [ ]:
ridge_train_pred = ridge_model.predict(X_train_ridge)
ridge_val_pred = ridge_model.predict(X_val_ridge)
ridge_test_pred = ridge_model.predict(X_test_ridge)

evaluate_model(y_train_ridge,ridge_train_pred,"Train")
evaluate_model(y_val_ridge,ridge_val_pred,"Validation")
evaluate_model(y_test_ridge,ridge_test_pred,"Test")

In [ ]:
ridge_result = pd.DataFrame([{
    "Model": "Ridge",
    "Test MAE": mean_absolute_error(y_test_ridge, ridge_test_pred),
    "Test RMSE": mean_squared_error(y_test_ridge, ridge_test_pred) ** 0.5,
    "Test R²": r2_score(y_test_ridge, ridge_test_pred)
}])

model_comparison = pd.concat(
    [model_comparison, ridge_result],
    ignore_index=True
)

model_comparison.sort_values("Test RMSE")

Ridge produced similar validation performance across the tested alpha values. It provides a useful linear comparison to the tree-based models, while also handling scaling, missing values, and categorical encoding through the preprocessing pipeline.

Random Forest was also tested during experimentation but performed worse than the other models, so it was not included in the final comparison.

### 5.7 XGBoost
XGBoost is tested as another gradient-boosting model using the numeric features with one-hot encoded location information.

In [ ]:
xgb_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

xgb_categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

xgb_preprocessor = ColumnTransformer([
    ("num", xgb_numeric_pipeline, hist_features),
    ("cat", xgb_categorical_pipeline, categorical_features_ridge)
])

In [ ]:
xgb_features = (
    hist_features
    + categorical_features_ridge
)

X_train_xgb = train_df[xgb_features]
y_train_xgb = train_df["target_temperature"]

X_val_xgb = val_df[xgb_features]
y_val_xgb = val_df["target_temperature"]

X_test_xgb = test_df[xgb_features]
y_test_xgb = test_df["target_temperature"]

In [ ]:
xgb_model = Pipeline([
    ("preprocessor", xgb_preprocessor),

    ("model", XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))])
xgb_model.fit(
    X_train_xgb,
    y_train_xgb
)

In [ ]:
xgb_train_pred = xgb_model.predict(X_train_xgb)
xgb_val_pred = xgb_model.predict(X_val_xgb)
xgb_test_pred = xgb_model.predict(X_test_xgb)

evaluate_model(y_train_xgb,xgb_train_pred,"Train")
evaluate_model(y_val_xgb,xgb_val_pred,"Validation")
evaluate_model(y_test_xgb,xgb_test_pred,"Test")

In [ ]:
xgb_result = pd.DataFrame([{
    "Model": "XGBoost",
    "Test MAE": mean_absolute_error(y_test_xgb, xgb_test_pred),
    "Test RMSE": mean_squared_error(y_test_xgb, xgb_test_pred) ** 0.5,
    "Test R²": r2_score(y_test_xgb, xgb_test_pred)
}])

model_comparison = pd.concat(
    [model_comparison, xgb_result],
    ignore_index=True
)

model_comparison.sort_values("Test RMSE")

### 5.8 Expanded Temperature History

The earlier feature importance results showed that recent temperature history was the strongest part of the forecast. Additional lag features and short-term temperature summaries were therefore tested.

In [ ]:
catboost_history_features = catboost_features + [
    "lag_2",
    "lag_3",
    "lag_7",
    "temp_change_1d",
    "temp_mean_3d"
]

In [ ]:
X_train_history = train_df[catboost_history_features]
X_val_history = val_df[catboost_history_features]

y_train = train_df["target_temperature"]
y_val = val_df["target_temperature"]

In [ ]:
catboost_history = CatBoostRegressor(
    iterations=1000,
    depth=6,
    learning_rate=0.10,
    l2_leaf_reg=3,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

catboost_history.fit(
    X_train_history,
    y_train,
    cat_features=["country", "location_name"],
    eval_set=(X_val_history, y_val),
    early_stopping_rounds=50
)

In [ ]:
history_train_pred = catboost_history.predict(X_train_history)
history_val_pred = catboost_history.predict(X_val_history)

evaluate_model(y_train,history_train_pred,"Train")
evaluate_model(y_val,history_val_pred,"Validation")

In [ ]:
history_importance = pd.DataFrame({
    "feature": catboost_history_features,
    "importance": catboost_history.get_feature_importance()
}).sort_values("importance", ascending=False)

history_importance

### 5.9 Final CatBoost Model
The final CatBoost model combines the original weather features with the additional temperature-history features that were tested above.

In [ ]:
final_catboost_features = [
    "temperature_celsius",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_7",
    "temp_change_1d",
    "temp_mean_3d",

    "humidity",
    "pressure_mb",
    "wind_kph",
    "log_precip_mm",
    "cloud",
    "uv_index",

    "day_of_year_sin",
    "day_of_year_cos",

    "latitude",
    "longitude",
    "country"
]

In [ ]:
X_train_final = train_df[final_catboost_features]
X_val_final = val_df[final_catboost_features]

y_train = train_df["target_temperature"]
y_val = val_df["target_temperature"]

In [ ]:
final_catboost = CatBoostRegressor(
    iterations=1000,
    depth=6,
    learning_rate=0.10,
    l2_leaf_reg=3,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

final_catboost.fit(
    X_train_final,
    y_train,
    cat_features=["country"],
    eval_set=(X_val_final, y_val),
    early_stopping_rounds=50
)

In [ ]:
final_train_pred = final_catboost.predict(X_train_final)
final_val_pred = final_catboost.predict(X_val_final)

evaluate_model(y_train,final_train_pred,"Train")
evaluate_model(y_val,final_val_pred,"Validation")

In [ ]:
X_test_final = test_df[final_catboost_features]
y_test = test_df["target_temperature"]

final_test_pred = final_catboost.predict(X_test_final)

evaluate_model(y_test,final_test_pred,"Test")

The final feature set keeps the variables that were useful in the earlier experiments while removing `location_name`, which contributed very little in the feature-importance analysis.

In [ ]:
final_importance = pd.DataFrame({
    "feature": final_catboost_features,
    "importance": final_catboost.get_feature_importance()
}).sort_values("importance", ascending=False)

In [ ]:
plt.figure(figsize=(10, 7))

sns.barplot(
    data=final_importance,
    x="importance",
    y="feature"
)

plt.title("Final CatBoost Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
explainer = shap.TreeExplainer(final_catboost)

shap_values = explainer.shap_values(X_val_final)

shap.summary_plot(
    shap_values,
    X_val_final,
    feature_names=final_catboost_features
)

In [ ]:
final_catboost_result = pd.DataFrame([{
    "Model": "Final CatBoost",
    "Test MAE": mean_absolute_error(y_test, final_test_pred),
    "Test RMSE": mean_squared_error(y_test, final_test_pred) ** 0.5,
    "Test R²": r2_score(y_test, final_test_pred)
}])

model_comparison = pd.concat(
    [model_comparison, final_catboost_result],
    ignore_index=True
)

model_comparison.sort_values("Test RMSE")

### 5.10 Ensemble
The final CatBoost model is combined with the other models to test whether their different predictions provide a more accurate forecast.

In [ ]:
ensemble_results = []
ensemble_cat_hist = (final_val_pred+ hist_val_pred) / 2
ensemble_results.append({
    "Ensemble": "CatBoost + HistGB",
    "MAE": mean_absolute_error(y_val, ensemble_cat_hist),
    "RMSE": mean_squared_error(y_val, ensemble_cat_hist) ** 0.5,
    "R²": r2_score(y_val, ensemble_cat_hist)
})

In [ ]:
ensemble_cat_xgb = (final_val_pred+ xgb_val_pred) / 2

ensemble_results.append({
    "Ensemble": "CatBoost + XGBoost",
    "MAE": mean_absolute_error(y_val, ensemble_cat_xgb),
    "RMSE": mean_squared_error(y_val, ensemble_cat_xgb) ** 0.5,
    "R²": r2_score(y_val, ensemble_cat_xgb)
})

In [ ]:
ensemble_cat_ridge = (final_val_pred+ ridge_val_pred) / 2

ensemble_results.append({
    "Ensemble": "CatBoost + Ridge",
    "MAE": mean_absolute_error(y_val, ensemble_cat_ridge),
    "RMSE": mean_squared_error(y_val, ensemble_cat_ridge) ** 0.5,
    "R²": r2_score(y_val, ensemble_cat_ridge)
})

In [ ]:
ensemble_three = (final_val_pred+ hist_val_pred+ xgb_val_pred) / 3

ensemble_results.append({
    "Ensemble": "CatBoost + HistGB + XGBoost",
    "MAE": mean_absolute_error(y_val, ensemble_three),
    "RMSE": mean_squared_error(y_val, ensemble_three) ** 0.5,
    "R²": r2_score(y_val, ensemble_three)
    })

In [ ]:
ensemble_four = (final_val_pred+ hist_val_pred+ xgb_val_pred+ ridge_val_pred) / 4

ensemble_results.append({
    "Ensemble": "All Four",
    "MAE": mean_absolute_error(y_val, ensemble_four),
    "RMSE": mean_squared_error(y_val, ensemble_four) ** 0.5,
    "R²": r2_score(y_val, ensemble_four
    )})

In [ ]:
ensemble_results_df = pd.DataFrame(ensemble_results)
ensemble_results_df.sort_values("RMSE")

In [ ]:
catboost_reference = pd.DataFrame([{
    "Ensemble": "CatBoost Alone",
    "MAE": mean_absolute_error(y_val, final_val_pred),
    "RMSE": mean_squared_error(y_val, final_val_pred) ** 0.5,
    "R²": r2_score(y_val, final_val_pred)
}])

pd.concat(
    [catboost_reference, ensemble_results_df],
    ignore_index=True
).sort_values("RMSE")

In [ ]:
weights = [
    (0.6, 0.4),
    (0.7, 0.3),
    (0.8, 0.2)
]

weighted_results = []

for cat_w, ridge_w in weights:
    pred = (
        cat_w * final_val_pred
        + ridge_w * ridge_val_pred
    )

    weighted_results.append({
        "CatBoost Weight": cat_w,
        "Ridge Weight": ridge_w,
        "MAE": mean_absolute_error(y_val, pred),
        "RMSE": mean_squared_error(y_val, pred) ** 0.5,
        "R²": r2_score(y_val, pred)
    })

pd.DataFrame(weighted_results).sort_values("RMSE")

In [ ]:
weighted_results_df = pd.DataFrame(weighted_results)

best_row = weighted_results_df.sort_values("RMSE").iloc[0]

best_cat_weight = best_row["CatBoost Weight"]
best_ridge_weight = best_row["Ridge Weight"]

print("Best CatBoost weight:", best_cat_weight)
print("Best Ridge weight:", best_ridge_weight)

In [ ]:
final_ensemble_test_pred = (
    best_cat_weight * final_test_pred
    + best_ridge_weight * ridge_test_pred
)

evaluate_model(
    y_test,
    final_ensemble_test_pred,
    "Final Ensemble Test"
)

In [ ]:
ensemble_result = pd.DataFrame([{
    "Model": "CatBoost + Ridge Ensemble",
    "Test MAE": mean_absolute_error(y_test, final_ensemble_test_pred),
    "Test RMSE": mean_squared_error(y_test, final_ensemble_test_pred) ** 0.5,
    "Test R²": r2_score(y_test, final_ensemble_test_pred)
}])

model_comparison = pd.concat(
    [model_comparison, ensemble_result],
    ignore_index=True
)

model_comparison.sort_values("Test RMSE")

### 5.11 Error Analysis


In [ ]:
error_df = test_df.copy()

error_df["prediction"] = final_ensemble_test_pred
# if +ve that would be underprediction , -ve overprediction
error_df["residual"] = (
    error_df["target_temperature"]
    - error_df["prediction"]
)

error_df["absolute_error"] = error_df["residual"].abs()

In [ ]:
error_df["next_day_change"] = (
    error_df["target_temperature"]
    - error_df["temperature_celsius"]
).abs()

pd.qcut(
    error_df["next_day_change"],
    q=4,
    duplicates="drop"
)

In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(
    y_test,
    final_ensemble_test_pred,
    alpha=0.2
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle="--"
)

plt.xlabel("Actual Temperature (°C)")
plt.ylabel("Predicted Temperature (°C)")
plt.title("Actual vs Predicted Temperature")
plt.tight_layout()
plt.show()

The predictions follow the actual temperatures closely, with most observations concentrated around the diagonal line. This is consistent with the high test R² of the final ensemble.

The spread becomes larger for some observations at the lower and higher ends of the temperature range, indicating that the model is less precise for some extreme temperatures.

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    error_df["residual"],
    bins=50
)

plt.xlabel("Residual (°C)")
plt.ylabel("Frequency")
plt.title("Distribution of Forecast Errors")
plt.tight_layout()
plt.show()

In [ ]:
error_df["change_group"] = pd.qcut(
    error_df["next_day_change"],
    q=4,
    duplicates="drop"
)

In [ ]:
change_error = (
    error_df.groupby("change_group", observed=True)
    .agg(
        observations=("absolute_error", "size"),
        mean_temp_change=("next_day_change", "mean"),
        MAE=("absolute_error", "mean"),
        RMSE=("residual", lambda x: np.sqrt(np.mean(x ** 2)))
    )
    .reset_index()
)

change_error

#### Error by Next-Day Temperature Change

The model is much more accurate when the next-day temperature is close to the current temperature. MAE increases from 0.50°C for the smallest changes to 2.99°C when the temperature change is above 2°C.

This suggests that the model benefits from the strong persistence of temperature, but has more difficulty capturing larger day-to-day changes. This is also consistent with the importance of current and recent temperature history in the final model.

In [ ]:
print("Mean residual:", error_df["residual"].mean())
print("Median residual:", error_df["residual"].median())

The residual distribution is centered close to zero, which indicates that the final ensemble does not have a large overall systematic bias.

The mean residual is slightly negative, while the median is close to zero. Since residuals are defined as actual minus predicted, the negative mean indicates a small tendency for the model to overpredict temperature.

In [ ]:
error_df["change_direction"] = np.select(
    [
        error_df["target_temperature"] > error_df["temperature_celsius"],
        error_df["target_temperature"] < error_df["temperature_celsius"]
    ],
    [
        "Warming",
        "Cooling"
    ],
    default="Little or no change"
)

direction_error = (
    error_df
    .groupby("change_direction", observed=True)
    .agg(
        observations=("absolute_error", "size"),
        MAE=("absolute_error", "mean"),
        RMSE=(
            "residual",
            lambda x: np.sqrt(np.mean(x ** 2))
        ),
        mean_residual=("residual", "mean")
    )
    .reset_index()
)

direction_error

The model behaves differently depending on the direction of the temperature change. During cooling events, the mean residual is -1.41°C, meaning that the model tends to overpredict the next-day temperature. During warming events, the mean residual is +1.08°C, meaning that it tends to underpredict the next-day temperature.

This suggests that the model tends to underestimate the magnitude of temperature changes and pull predictions toward the current temperature. When there is little or no temperature change, the error is much smaller, with an MAE of only 0.49°C.

In [ ]:
error_df["lag_1_available"] = error_df["lag_1"].notna()

lag_error = (
    error_df.groupby("lag_1_available")
    .agg(
        observations=("absolute_error", "size"),
        MAE=("absolute_error", "mean"),
        RMSE=("residual", lambda x: np.sqrt(np.mean(x ** 2)))
    )
    .reset_index())

lag_error

### Effect of Missing Recent History

The availability of `lag_1` has a noticeable effect on forecast accuracy. When the previous day's temperature is available, the ensemble achieves an MAE of 1.34°C, compared with 2.06°C when `lag_1` is missing.

This supports the importance of recent temperature history for the forecasting task. However, only 231 of the 25,000 test observations are missing `lag_1`, so this result represents a small subset of the test set.

In [ ]:
error_df["temperature_group"] = pd.qcut(
    error_df["target_temperature"],
    q=5,
    duplicates="drop"
)

temperature_error = (
    error_df
    .groupby("temperature_group", observed=True)
    .agg(
        observations=("absolute_error", "size"),
        mean_actual_temperature=("target_temperature", "mean"),
        MAE=("absolute_error", "mean"),
        mean_residual=("residual", "mean"),
        RMSE=(
            "residual",
            lambda x: np.sqrt(np.mean(x ** 2))
        )
    )
    .reset_index()
)

temperature_error

The error also varies with the actual temperature. The lowest-temperature group has a mean residual of -1.00°C, while the highest-temperature group has a mean residual of +0.53°C. Since residuals are defined as actual minus predicted, this indicates that colder temperatures tend to be overpredicted and hotter temperatures tend to be underpredicted.

The model is most accurate in the middle temperature ranges, where the MAE falls below 1.1°C. Together with the warming and cooling analysis, this suggests that the model tends to produce more conservative predictions and has difficulty reaching more extreme temperatures.

### Error Analysis Summary

The final ensemble performs best when the next-day temperature remains relatively stable. Its error increases as the magnitude of the temperature change increases.

The direction and temperature-group analyses show a consistent pattern: cooling temperatures tend to be overpredicted, while warming and hotter temperatures tend to be underpredicted. This indicates that the model tends to dampen larger temperature changes and pull predictions toward the middle of the observed temperature range.

The availability of `lag_1` also has a noticeable effect on performance, with substantially lower error when recent temperature history is available. Overall, the error analysis shows that the main limitation is not general prediction accuracy, but capturing larger and more extreme day-to-day temperature changes.

#### Future Improvment Experiment
Maybe predicting the change directly will stop the model from pulling predictions toward today's temperature.

### 5.12 Saving the Final Models

In [ ]:
final_catboost.save_model("final_catboost.cbm")

joblib.dump(
    ridge_model,
    "ridge_pipeline.joblib"
)

joblib.dump(
    {
        "catboost_weight": best_cat_weight,
        "ridge_weight": best_ridge_weight
    },
    "ensemble_weights.joblib"
)